# Day 4 — Reliability, Human-in-the-Loop & Cost Governance

---

Agents that work in demos crash in production. Today's material is the unsexy stuff that separates a toy project from something you can put your name on:

1. **Retries + error recovery** — tools fail; agents shouldn't die
2. **Human-in-the-loop (HITL)** — pause before doing anything irreversible
3. **Cost governance** — max iterations, per-run token budgets

If you skip these, an agent can happily rack up hundreds of dollars in API costs or fire off a `DELETE /users/*` in an infinite loop. Ask anyone who's shipped an agent — they've been burned.


## 1. Tool failures — retry with backoff

Real tools fail: network hiccups, rate limits, 503s. Wrap tool calls with a retry helper.


In [1]:
import time
import functools

def with_retries(fn, max_attempts: int = 3, base_delay: float = 0.5):
    @functools.wraps(fn)
    def wrapper(*args, **kwargs):
        last_err = None
        for attempt in range(max_attempts):
            try:
                return fn(*args, **kwargs)
            except Exception as e:
                last_err = e
                time.sleep(base_delay * (2 ** attempt))   # 0.5, 1, 2, ...
        return f"tool failed after {max_attempts} attempts: {last_err}"
    return wrapper


@with_retries
def flaky_tool(x: str) -> str:
    import random
    if random.random() < 0.7:
        raise RuntimeError("network glitch")
    return f"processed: {x}"

for _ in range(3):
    print(flaky_tool("hello"))


tool failed after 3 attempts: network glitch
processed: hello
processed: hello


**Key rule:** *retry on transient failures, not on logic errors.* A 500 from a downstream API is transient. A `ValueError: bad input` from your own code is a bug — retrying just wastes time. In production, filter which exceptions retry (`requests.HTTPError`, `httpx.TimeoutException`, etc.).


## 2. Cost governance — the three limits every agent needs

Every agent run should be bounded by:

1. **max_steps** — hard cap on tool-loop iterations
2. **max_tokens_per_run** — count input + output tokens and stop if you exceed
3. **max_cost_per_run** — same idea, but converted to dollars

These prevent the runaway-cost horror stories.


In [2]:
import tiktoken

enc = tiktoken.encoding_for_model("gpt-4o-mini")
PRICE_PER_MTOK = 0.10   # blended $/1M for openai/gpt-oss-20b (~$0.05 in + $0.20 out, ~70/30 mix)

class Budget:
    def __init__(self, max_steps: int = 8, max_tokens: int = 20_000,
                 max_usd: float = 0.10):
        self.max_steps = max_steps
        self.max_tokens = max_tokens
        self.max_usd = max_usd
        self.steps = 0
        self.tokens = 0

    def charge_step(self) -> None:
        self.steps += 1
        if self.steps > self.max_steps:
            raise BudgetExceeded("max_steps")

    def charge_tokens(self, text: str) -> None:
        self.tokens += len(enc.encode(text))
        if self.tokens > self.max_tokens:
            raise BudgetExceeded("max_tokens")
        if (self.tokens / 1_000_000) * PRICE_PER_MTOK > self.max_usd:
            raise BudgetExceeded("max_usd")


class BudgetExceeded(Exception):
    pass


b = Budget(max_steps=3, max_tokens=100, max_usd=1.0)
try:
    for i in range(10):
        b.charge_step()
        b.charge_tokens("some prompt text that adds up")
        print(f"step {i} ok — tokens so far: {b.tokens}")
except BudgetExceeded as e:
    print(f"HALT: {e}")


step 0 ok — tokens so far: 6
step 1 ok — tokens so far: 12
step 2 ok — tokens so far: 18
HALT: max_steps


**Put the budget at the top of your agent loop.** Every iteration calls `budget.charge_step()`; every LLM call increments `budget.charge_tokens(prompt + response)`. When either limit is hit, the run stops and you return a partial result — not a black hole of cost.


## 3. Human-in-the-loop — the "are you sure?" pattern

Some tool calls are **irreversible**:

- Sending an email
- Executing a database `DELETE`
- Making a payment
- Posting to Slack

The pattern: **before executing a "dangerous" tool, pause and ask a human to approve.** LangGraph has first-class support for this with `interrupt_before=`. Or you can implement it manually.


In [3]:
DANGEROUS = {"send_email", "delete_row", "charge_card"}

def run_tool_with_approval(name: str, args: dict, tools: dict) -> str:
    if name in DANGEROUS:
        # In a real app, this is a WebSocket message to a UI, or a
        # message-queue task waiting for a human's response.
        # For the demo, we simulate with input().
        prompt = f"HITL: agent wants to call {name}({args}). Approve? [y/N] "
        try:
            ok = input(prompt).strip().lower() == "y"
        except EOFError:
            ok = False
        if not ok:
            return "denied by human"
    return tools[name](**args)


In production, you'd:

- Store the pending call in a database with a status of `"awaiting_approval"`.
- Send a Slack / email / dashboard notification to a reviewer.
- When they click Approve, resume the agent from that checkpoint.

LangGraph's `Checkpointer` + `interrupt_before` do this natively — worth reading their docs when your agent needs it.


## 4. Async background tasks — FastAPI's built-in tool

Agents can take 10–60 seconds. You **cannot** make an HTTP request wait that long. The fix: hand the agent to a background task and return a job ID.

For freshers, **FastAPI's `BackgroundTasks`** is enough. It's built in, requires zero setup.

```python
from fastapi import FastAPI, BackgroundTasks

app = FastAPI()
JOBS = {}   # jobs["abc"] = {"status": "running", "result": None}

def run_agent_job(job_id: str, question: str):
    try:
        result = agent(question)              # your slow agent
        JOBS[job_id] = {"status": "done", "result": result}
    except Exception as e:
        JOBS[job_id] = {"status": "failed", "error": str(e)}

@app.post("/agent")
def start_agent(question: str, bg: BackgroundTasks):
    job_id = os.urandom(4).hex()
    JOBS[job_id] = {"status": "running"}
    bg.add_task(run_agent_job, job_id, question)
    return {"job_id": job_id}

@app.get("/agent/{job_id}")
def status(job_id: str):
    return JOBS.get(job_id, {"status": "unknown"})
```

**When to upgrade to Celery + Redis:** when jobs need to survive server restarts, be retried automatically, or scale across multiple worker machines. For a single-server side project: `BackgroundTasks` is fine.


## 5. Putting it all together — a "responsible" agent


In [4]:
from together import Together
from dotenv import load_dotenv
import os, json
load_dotenv()
llm = Together()

def responsible_agent(question: str,
                      tools: dict,
                      tool_schemas: list,
                      budget: Budget) -> str:
    messages = [
        {"role": "system", "content": "You are a helpful assistant. Use tools as needed."},
        {"role": "user",   "content": question},
    ]
    while True:
        try:
            budget.charge_step()
        except BudgetExceeded as e:
            return f"stopped: budget ({e})"

        resp = llm.chat.completions.create(
            model="openai/gpt-oss-20b",
            messages=messages, tools=tool_schemas, tool_choice="auto",
            temperature=0.0,
        )
        msg = resp.choices[0].message
        try:
            budget.charge_tokens(msg.content or "")
        except BudgetExceeded as e:
            return f"stopped: budget ({e})"

        if not msg.tool_calls:
            return msg.content or "(empty)"

        messages.append({"role": "assistant", "content": msg.content or "",
                         "tool_calls": [tc.model_dump() for tc in msg.tool_calls]})
        for tc in msg.tool_calls:
            name = tc.function.name
            args = json.loads(tc.function.arguments)
            obs = tools[name](**args) if name in tools else f"unknown {name}"
            messages.append({"role": "tool", "tool_call_id": tc.id,
                             "name": name, "content": obs})

print("Ready — supply tools/schemas from Day 2 and try it.")


Ready — supply tools/schemas from Day 2 and try it.


## Recap

- Wrap flaky tools with **retries and backoff**. Retry transient errors only.
- Every agent needs three budgets: **max_steps, max_tokens, max_usd**. Check at the top of the loop.
- **HITL:** stop before irreversible actions and require human approval. Log the request either way.
- **FastAPI `BackgroundTasks`** for slow agents in a web API. Upgrade to Celery + Redis only when you need durability/scaling.
- **Next class:** the capstone — a real autonomous research assistant that puts every piece together.
